# ③ 검증 — 평가 데이터셋과 골든 근거 라벨
담당: 검증 · 코드: `rag/validation/validation.py`, `rag/validation/evidence.py`

골든 dev 408문항과 RFP 평가 질문을 점검해 `artifacts/eval_set.jsonl`에 저장합니다.
골든 test 943문항은 라벨 품질 보고서에서만 읽고 평가셋에는 넣지 않습니다. 검색·생성·API 호출은 없습니다.


In [1]:
import pandas as pd

from rag.config import ARTIFACTS, ROOT, load_config, read_jsonl, write_jsonl
from rag.validation.validation import check_eval_items, load_eval_items
from rag.validation.evidence import (
    correspondence_report, document_report, label_evidence, labelled_sample, mapping_report,
)

cfg = load_config()
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
raw_dir = ARTIFACTS / "raw"
input_path = ROOT / cfg["paths"]["input"]
package_dir = ROOT / "data/RFP_Preprocessing_Final_v11_Team"


## 근거 항목과 [E] 블록의 대응

`context`에는 인용하지 않은 블록도 있고, `evidence`는 선택한 근거 참조만 담습니다.
순서대로 짝짓지 않습니다. v11 `final_elements.jsonl`의 `element_id`와 `location`으로
각 항목의 본문을 복원합니다. 글은 요소의 `[start:end]`, 표는 해당 행의 칸을 열 순서로 ` | `로 잇습니다.
복원한 본문이 context 블록 전체와 같은지 확인하고 모든 해당 `[E]` 라벨을 보관합니다.
이 좌표는 본문 복원에만 쓰며 현재 청크의 좌표로 변환하지 않습니다.

파서 문서는 설정한 원본 폴더·파일(ZIP은 `artifacts/raw`에 푼 파일)의 SHA-256 → NFC 파일명 → 현재 `source/doc_id`로 연결합니다.
v11은 패키지의 원본 SHA와 현재 `doc_id`·`source`가 모두 일치할 때 연결하므로 raw 폴더가 필요하지 않습니다.
NFC 정규화, 표 파이프를 공백으로 바꾸기, 연속 공백 축약 후 근거 본문이 같은 문서의
청크 `body`에 부분 문자열로 들어가야 합니다. 유일한 청크 ID일 때만 라벨을 붙입니다.
20자 미만은 짧은 수·코드·항목 이름만으로 청크를 확정하지 않도록 `too_short`로 둡니다.
문서 미연결을 먼저 판정하며, 나머지는 `not_found`, `ambiguous`입니다. 유사도 점수는 사용하지 않습니다.


In [2]:
items = load_eval_items(cfg["validation"]["golden_sets"])
labelled_items = label_evidence(items, chunks, raw_dir, package_dir, input_path)
dev_items = [item for item in labelled_items if item["evidence_source"] == "golden"]
test_input = load_eval_items(["eval/golden/test.jsonl"])
test_items = label_evidence(test_input, chunks, raw_dir, package_dir, input_path)
splits = {"dev": dev_items, "test": test_items}

audit = pd.DataFrame([
    {"split": "dev", **correspondence_report(
        [it for it in items if it["evidence_source"] == "golden"], dev_items, chunks)},
    {"split": "test", **correspondence_report(test_input, test_items, chunks)},
])
audit.to_csv(ARTIFACTS / "golden_correspondence.csv", index=False)
display(audit)


,split,questions,evidence_entries,context_blocks,entries_with_context_text,entries_with_repeated_context_text,original_chunk_ids_live,minimum_normalized_length
0,dev,408,716,1400,716,14,0,20
1,test,943,1639,3245,1639,26,0,20


## 문서 연결과 미연결 목록

`unanswerable`의 인용 근거는 다른 사업 문서의 예시이므로 질문 대상 문서의 정답 근거로 해석하지 않습니다.
이 보고서는 원래 인용한 근거의 소속만 연결합니다. 따라서 dev 질문 대상 30문서와 근거의 36 SHA는 다릅니다.


In [3]:
documents = document_report(splits)
documents.to_csv(ARTIFACTS / "golden_documents.csv", index=False)
display(documents.groupby(["split", "outcome"]).size().unstack(fill_value=0))
display(documents[documents["outcome"] == "doc_unmapped"])


outcome,doc_unmapped,matched
split,,
dev,1,35
test,1,69


,split,doc_sha,source,doc_id,golden_source,outcome
8,dev,3b643668220fa1ccaec14adea446bb8b7c604c46dfd9b9...,None,None,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,doc_unmapped
65,test,73abf8e4f78c36a19e2345a3e2a0be5afd0e8a5a1d2736...,None,None,중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp,doc_unmapped


## 주제별 라벨 품질 — 검색 없이 dev · test 전체

각 행은 근거 항목 수와 결과별 개수, 모든 근거·일부 근거·어느 근거도 청크 라벨이 없는 질문 수입니다.
`multi_hop`과 `comparison`을 따로 표시합니다. 03의 핵심 사실 점검에서 빠지는 질문도 여기에는 포함합니다.
`document_rate`, `chunk_rate`의 분모는 각각 해당 행의 전체 근거 항목 수입니다.


In [4]:
mapping = mapping_report(splits)
mapping.to_csv(ARTIFACTS / "golden_mapping.csv", index=False)
display(mapping)


,split,primary_topic,questions,evidence_entries,labelled,doc_unmapped,not_found,ambiguous,too_short,fully_labelled,partly_labelled,not_labelled,chunk_rate,document_rate
0,dev,ALL,408,716,306,24,64,32,290,140,125,143,0.427374,0.966480
1,dev,comparison,56,137,61,7,3,5,61,9,31,16,0.445255,0.948905
2,dev,condition,69,74,57,2,14,1,0,52,2,15,0.770270,0.972973
3,dev,false_premise,4,7,3,0,1,0,3,0,3,1,0.428571,1.000000
4,dev,multi_hop,71,190,48,8,14,16,104,5,30,36,0.252632,0.957895
5,dev,numeric,104,105,70,3,29,2,1,69,0,35,0.666667,0.971429
6,dev,table,96,192,63,4,2,8,115,3,57,36,0.328125,0.979167
7,dev,unanswerable,8,11,4,0,1,0,6,2,2,4,0.363636,1.000000
8,test,ALL,943,1639,708,19,188,50,674,328,255,360,0.431971,0.988408
9,test,comparison,128,291,142,6,21,13,109,31,54,43,0.487973,0.979381


## 평가 항목 점검 · 저장

dev와 YAML 항목만 저장합니다. YAML은 `evidence_source = "none"`, `doc_ids = []`, `evidence = []`로 표시합니다.
골든 항목에는 `doc_ids`와 각 근거의 본문·현재 `doc_id`·유일한 `chunk_id` 또는 사유를 저장합니다.
`golden_chunk_id`에는 원래 ID를 남겨 현재 라벨과 구분합니다.


In [5]:
eval_set, report = check_eval_items(labelled_items)
write_jsonl(eval_set, ARTIFACTS / "eval_set.jsonl")
print(f"평가 데이터셋 {len(eval_set)}개 / 점검 {len(items)}개")
display(report["점검"].value_counts())
display(report[report["점검"] != "통과"])


평가 데이터셋 385개 / 점검 416개


점검
통과                                        385
핵심 사실(key_facts) 없음                        24
후속 질문 — 04 생성은 질문마다 새 대화라 앞 질문의 맥락이 없음      4
정답(reference) 없음                            3
Name: count, dtype: int64

,id,질문,점검
325,multi_hop-A-night1-r1-0020,경기도 평택시의 「2024년도 평택시 버스정보시스템(BIS) 구축사업」 사업에서 발...,핵심 사실(key_facts) 없음
327,multi_hop-A-night1-r1-0087,인천광역시의 「인천일자리플랫폼 정보시스템 구축 ISP 수립용역」 사업에서 플랫폼의 ...,핵심 사실(key_facts) 없음
330,multi_hop-A-night1-r1-0263,한국로봇산업진흥원의 「한국로봇산업진흥원 사업관리시스템 온라인평가」 사업에서 계약 후...,핵심 사실(key_facts) 없음
333,multi_hop-A-night1-r1-0288,대전대학교의 「대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전」...,핵심 사실(key_facts) 없음
336,multi_hop-A-regen1-r1-0080,경상북도 봉화군의 「봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)」 사업에서 ...,핵심 사실(key_facts) 없음
341,multi_hop-A-regen1-r3-0053,한국생산기술연구원의 「EIP3.0 고압가스 안전관리 시스템 구축 용역」 사업에서 구...,핵심 사실(key_facts) 없음
343,multi_hop-A-span1-r1-0040,고려대학교의 「차세대 포털·학사 정보시스템 구축사업」 사업에서 채용시스템과 교원초빙...,핵심 사실(key_facts) 없음
350,multi_hop-A-span2-r1-0003,한국철도공사의 「운행정보기록 자동분석시스」 사업에서 산출정보에 '시스템 매뉴얼'이 ...,핵심 사실(key_facts) 없음
361,multi_hop-A-span2-r2-0043,한국생산기술연구원의 「EIP3.0 고압가스 안전관리 시스템 구축 용역」 사업에서 문...,핵심 사실(key_facts) 없음
362,multi_hop-A-span2-r2-0044,대전대학교의 「대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전」...,핵심 사실(key_facts) 없음


## 고정 seed 42 — 라벨된 dev 근거 30쌍 대조

질문 ID·청크 ID·근거 본문 순으로 정렬한 라벨된 근거에서 최대 30항목을 비복원 추출합니다. 30쌍 미만이면 전부, 0쌍이면 빈 표를 표시합니다.
왼쪽은 골든 근거 본문, 오른쪽은 연결한 청크의 전체 `body`입니다. 가로로 나란히 읽으며 확인합니다.


In [6]:
sample = labelled_sample(dev_items, chunks)
display(sample.style.format(escape="html").set_properties(**{
    "white-space": "pre-wrap", "text-align": "left", "vertical-align": "top",
}).set_table_styles([
    {"selector": "td", "props": [("min-width", "160px"), ("max-width", "540px")]},
]))


,question_id,doc_id,chunk_id,golden_evidence_text,matched_chunk_text
0,comparison-A-regen1-r1-0019,서영대학교_산학협력단_전문대학_혁신지원사업_서영대학교_차세대_교육_579ff4,서영대학교_산학협력단_전문대학_혁신지원사업_서영대학교_차세대_교육_579ff4::c00081,프로젝트 관리 요구사항(PMR) | 프로젝트의 원활한 수행을 위한 관리 방법 및 추진 단계별 수행 방안에 대한 요구사항 | 18,"| 분류기준 | 세 부 내 용 | 항목 | | --- | --- | --- | | 기능요구사항 (SFR) | 목표 시스템(사업)이 반드시 수행해야 하거나 목표시스템을 이용하여 사용자가 반드시 수행할 수 있어야하는 기능(동작)에 대한 요구사항 | 16 | | 성능요구사항 (PER) | 목표 시스템의 처리속도 및 시간, 처리량, 동적-정적 용량, 가용성 등 성능에 대한 요구사항 | 9 | | 인터페이스 요구사항 (SIR) | 목표 시스템과 외부를 연결하는 시스템 인터페이스와 사용자 인터페이스에 대한 요구사항을 기술한 것으로 타 소프트웨어, 하드웨어 및 통신 인터페이스, 타 시스템들과의 정보교환에 이용되는 프로토콜과의 연계도 포함하여 기술 | 5 | | 데이터 요구사항 (DAR) | 목표 시스템의 서비스에 필요한 초기자료 구축 및 데이터 변환을 위한 대상, 방법, 보안이 필요한 데이터 등 데이터를 구축하기 위해 필요한 요구사항 | 7 | | 테스트 요구사항 (TER) | 도입되는 장비의 성능 테스트 (BMT) 또는 구축된 시스템이 계획된 목표 대비 제대로 운영되는가를 점검하기 위한 테스트 요구사항 | 7 | | 보안 요구사항 (SER) | 정보 자산의 기밀성과 무결성을 확보하기 위해 목표 시스템의 데이터 및 기능, 운영 접근을 통제하기 위한 요구사항 | 7 | | 품질 요구사항 (QUR) | 목표 사업의 원활한 수행 및 운영을 위해 관리가 필요한 품질 항목, 품질 평가 대상 및 목표에 대한 요구사항 | 11 | | 제약사항(COR) | 목표 시스템 설계, 구축, 운영과 관련하여 사전에 파악된 기술-표준-업무-법 제도 등 제약조건 | 9 | | 프로젝트 관리 요구사항(PMR) | 프로젝트의 원활한 수행을 위한 관리 방법 및 추진 단계별 수행 방안에 대한 요구사항 | 18 | | 프로젝트 지원 요구사항(PSR) | 프로젝트의 원활한 수행을 위해 필요한 지원사항 및 방안에 대한 요구사항 | 8 | | 프로젝트 지원 요구사항(PSR) | 합 계 | 97 |"
1,comparison-A-fill3-r1-0005,한국연구재단_2024년_대학산학협력활동_실태조사_시스템_UICC_기능개선_48ae20,한국연구재단_2024년_대학산학협력활동_실태조사_시스템_UICC_기능개선_48ae20::c00092,요구사항 명칭 | 확인 및 에러 메시지 제공,"| 요구사항 분류 | 인터페이스 요구사항 | | --- | --- | | 요구사항 고유번호 | INR-005 | | 요구사항 명칭 | 확인 및 에러 메시지 제공 | | 요구사항 상세설명 | 정의 | 확인 및 에러 메시지 제공 정의 | | 세부내용 | ㅇ 삭제, 입력 정보 완료 또는 미완료 후 저장 등 사용자의 수행 활동에 대한 메시지 제공 및 보완 ㅇ 에러 및 확인 메시지는 통일성 있게 처리 | | 산출정보 | |"
2,multi_hop-A-night1-r1-0058,세종테크노파크_세종테크노파크_인사정보_전산시스템_구축_용역_입찰공_6a3439,세종테크노파크_세종테크노파크_인사정보_전산시스템_구축_용역_입찰공_6a3439::c00059,요구사항 분류 | 요구사항 번호 | 요구사항 명칭,| 요구사항 분류 | 요구사항 번호 | 요구사항 명칭 | | --- | --- | --- | | 기능-SFR | SFR-001 | 인사시스템 일반사항 | | 기능-SFR | SFR-002 | 유연근무 도입을 통한 근태관리 기능 고도화 | | 성능-PER | PER-001 | 성능 일반 | | 성능-PER | PER-002 | 배치성 업무 응답시간 | | 성능-PER | PER-003 | 평균 응답시간 | | 성능-PER | PER-004 | 평균 처리시간 | | 성능-PER | PER-005 | 웹 사이트 최적화 | | 인터페이스-SIR/UIR | SIR-001 | 통합 UI | | 인터페이스-SIR/UIR | SIR-002 | 사용자 UI&UX | | 인터페이스-SIR/UIR | SIR-003 | 크로스브라우징 | | 데이터-DAR | DAR-001 | 데이터베이스 설계 기준 | | 테스트-TER | TER-001 | 단위테스트 | | 테스트-TER | TER-002 | 통합테스트 | | 테스트-TER | TER-003 | 인수테스트 | | 보안-SER | SER-001 | 보안지침 준수 일반사항 | | 보안-SER | SER-002 | 시큐어코딩 및 보안취약점 제거 | | 보안-SER | SER-003 | 개인정보보호 및 데이터 암호화 | | 보안-SER | SER-004 | 데이터 무결성 | | 보안-SER | SER-005 | 정보 누출금지 대상 | | 보안-SER | SER-006 | 수행업체 보안 준수 사항 | | 품질-QUR | QUR-001 | 품질관리 일반사항 | | 품질-QUR | QUR-002 | 가용성 보장 | | 품질-QUR | QUR-003 | 사용성 보장 | | 품질-QUR | QUR-004 | 상호 운용성 | | 제약사항-COR | COR-001 | 사업관련 공통 규정 준수 | | 제약사항-COR | COR-002 | 사업관련 공통 기술표준 준수 | | 제약사항-COR | COR-003 | 용역업체 정보보호 및 보안준수 | | 제약사항-COR | COR-004 | 정보시스템 관련 규정 및 표준 준수 | | 제약사항-COR | COR-005 | 웹 접근성 준수 | | 제약사항-COR | COR-006 | 웹 표준 준수 | | 제약사항-COR | COR-007 | 데이터표준화 기준 준수 | | 제약사항-COR | COR-008 | 개발언어 사용 | | 제약사항-COR | COR-009 | 기존 시스템과 호환성 적용 | | 제약사항-COR | COR-010 | 기타 제약사항 | | 프로젝트관리-PMR | PMR-001 | 프로젝트 관리 일반사항 | | 프로젝트관리-PMR | PMR-002 | 프로젝트관리 방법론 준용 등 사업관리 요건 준수 | | 프로젝트관리-PMR | PMR-003 | 산출물 작성 및 제출 | | 프로젝트관리-PMR | PMR-004 | 프로젝트 조직 구성 | | 프로젝트관리-PMR | PMR-005 | 개발방법론 적용 시 유의사항 | | 프로젝트관리-PMR | PMR-006 | 시험운영 일정 확보 | | 프로젝트지원-PSR | PSR-001 | 교육지원 | | 프로젝트지원-PSR | PSR-002 | 개발장소 지정 | | 프로젝트지원-PSR | PSR-003 | 기술이전 | | 프로젝트지원-PSR | PSR-004 | 하자보수 일반 | | 프로젝트지원-PSR | PSR-005 | 사업정보 저장소 데이터 관리 |
3,multi_hop-A-fill1-r1-0090,한국건강가정진흥원_2025년_아이돌봄인력_인적성_검사_정보시스템_운영_cc0530,한국건강가정진흥원_2025년_아이돌봄인력_인적성_검사_정보시스템_운영_cc0530::c00088,요구사항 상세설명 | 정의 | 품질을 위한 프로젝트 위험관리 요건,"| 요구사항 고유번호 | PMR-007 | | --- | --- | | 요구사항 명칭 | 리스크 관리 | | 요구사항 분류 | 프로젝트 관리 요구사항 | | 요구사항 상세설명 | 정의 | 품질을 위한 프로젝트 위험관리 요건 | | 세부 내용 | ㅇ 보안, 일정지연, 품질저하, 장비 변경에 따르는 예산초과 등 리스크 발생을 사전 예방하고 발생 시 사후 대처방안을 제시하여야 함 ㅇ 성능상 문제 등으로 클라우드 또는 S/W가 변경되는 경우에 대한 사후 대응방안을 구체적으로 제시 ㅇ 유지관리과정에서 요구되는 진척/위험/변경사항의 관리방안 및 지속적으로 문제를 파악 관리할 수 있는 방안 제시 |"
4,condition-